# Fine-tune Llama 3.1 8B Instruct on OSDG-CD with Unsloth (LoRA, 4-bit), then score OSDG and SDGi - Colab, web

In [ ]:
!nvidia-smi

Fri Oct 23 06:39:42 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA L4                      Off |   00000000:00:03.0 Off |                    0 |
| N/A   41C    P8             13W /   72W |       0MiB /  23034MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## Install (run once, then Runtime > Restart session)
Qwen3.5 needs `transformers` v5. Dependency-conflict warnings from pip are usually harmless.

In [ ]:
!pip install -q --upgrade unsloth unsloth_zoo "transformers>=5.0.0,<=5.17.0" scikit-learn tqdm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.8/79.8 kB 8.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.4/57.4 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.0/31.0 MB 89.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.7/2.7 MB 114.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 149.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 148.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.2/80.2 kB 8.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 63.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.8/73.8 kB 8.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 125.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 945.8/945.8 kB 66.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 108.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 M

## Extract the data exported from V9
Drag `qwen_export.zip` into the Files pane on the left first (the upload widget of `files.upload()` does not always accept zip files).

In [ ]:
import os, zipfile
assert os.path.exists('qwen_export.zip'), "Drag qwen_export.zip into the Files pane first. Files here: %s" % os.listdir('.')
zipfile.ZipFile('qwen_export.zip').extractall('qwen_export')
print(os.listdir('qwen_export'))

['sdgi_chunks_dev.parquet', 'osdg_test.parquet', 'sdgi_chunks_test.parquet', 'osdg_dev.parquet', 'robust_osdg_test_all.parquet', 'meta.json', 'robust_pert.parquet', 'osdg_train.parquet']


# The method (read once)

We do not generate text. The prompt asks the model to answer with ONE word (Poverty, Hunger, Health, ... Peace). We read the model's next-token logits for the first token of each of the 16 words, softmax them, and get a 16-column probability matrix. This is the "score matrix" the V9 notebook expects from every model (argmax for Setting A, threshold for Setting B). Training = cross-entropy over those 16 logits, so the loss targets exactly the quantity we score.

**Unsloth's role:** `from_pretrained` loads the model with fast kernels, `get_peft_model` attaches small LoRA adapters (only these are trained) and enables memory-saving gradient checkpointing. The training loop itself is plain PyTorch.

**Order:** (1) config, (2) data, (3) load model, (4) prompt + label tokens, (5) scoring helpers, (6) zero-shot baseline of the SAME untouched model, (7) add LoRA, (8) train + save adapter, (9) score everything + save.

In [ ]:
# STEP 0 - imports. `unsloth` must be imported BEFORE transformers / peft.
import os, sys, json, time, math, random, gc, shutil
from pathlib import Path
from unsloth import FastLanguageModel
import numpy as np, pandas as pd, torch, torch.nn.functional as F
from sklearn.metrics import accuracy_score, f1_score
from tqdm.auto import tqdm
from google.colab import files

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
Unsloth: xformers is installed but cannot load with torch 2.11.0+cu130 (it was built for 2.10.0+cu128), so PyTorch attention is used instead.
To install a matching xformers, run `unsloth install-kernels xformers`, or `pip uninstall xformers` to hide this. UNSLOTH_ENABLE_LOGGING=1 shows the full error.
🦥 Unsloth Zoo will now patch everything to make training faster!


## Step 1 - Config
Training settings are the ones of the Qwen3.5-9B run (the same 8B-9B class of model, 4-bit)

In [ ]:
CFG = dict(
    model="unsloth/Llama-3.1-8B-Instruct",   # Unsloth upload of meta-llama/Llama-3.1-8B-Instruct; loaded in 4-bit
    export_dir="qwen_export",
    out="llama31_8b_lora_osdg.npz",
    train_n=6000,                   # random sample (natural class distribution); 0 = all 15,984
    epochs=1,                       # with 2+ epochs the best one is chosen on a random dev sample
    lr=2e-4,                        # the value that worked in the first run (9B, 4-bit)
    bs=8,                           # micro-batch (lower it on CUDA out-of-memory)
    grad_acc=2,                     # effective batch = bs * grad_acc
    eval_bs=32,                     # max texts per scoring batch (also capped by a token budget)
    rank=16,                        # LoRA rank
    max_words=200,                  # each text is cut to its first N words
    max_seq=768,
    seed=42,
    quick=False,                      # FIRST RUN: True (sanity check). Then False for the real run
    load_in_4bit=True,              # as in the first run (about 8 GiB of weights)
    lora_fp32=False,                # True only for 16-bit runs: casts the LoRA weights to fp32
    label="Llama-3.1-8B-Instruct",  # display name for V9
)
# from google.colab import userdata; os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")   # only for the gated meta-llama/ repo
random.seed(CFG["seed"]); np.random.seed(CFG["seed"]); torch.manual_seed(CFG["seed"])
assert torch.cuda.is_available(), "No GPU. Runtime > Change runtime type > GPU, then run !nvidia-smi."
print("GPU:", torch.cuda.get_device_name(0), "| VRAM %.1f GiB" % (torch.cuda.get_device_properties(0).total_memory / 2**30))
print(CFG)

GPU: NVIDIA L4 | VRAM 22.0 GiB
{'model': 'unsloth/Llama-3.1-8B-Instruct', 'export_dir': 'qwen_export', 'out': 'llama31_8b_lora_osdg.npz', 'train_n': 6000, 'epochs': 1, 'lr': 0.0002, 'bs': 8, 'grad_acc': 2, 'eval_bs': 32, 'rank': 16, 'max_words': 200, 'max_seq': 768, 'seed': 42, 'quick': False, 'load_in_4bit': True, 'lora_fp32': False, 'label': 'Llama-3.1-8B-Instruct'}


## Step 2 - Data (exported from V9, so the splits are identical to the other models)

In [ ]:
EXP = Path(CFG["export_dir"])
meta = json.load(open(EXP / "meta.json"))
COMMON = [int(s) for s in meta["COMMON"]]; K = len(COMMON)
assert COMMON == list(range(1, 17)), "the label words below assume SDGs 1-16"
tr = pd.read_parquet(EXP / "osdg_train.parquet"); dv = pd.read_parquet(EXP / "osdg_dev.parquet"); te = pd.read_parquet(EXP / "osdg_test.parquet")
ch_dev = pd.read_parquet(EXP / "sdgi_chunks_dev.parquet"); ch_te = pd.read_parquet(EXP / "sdgi_chunks_test.parquet")
idx_A = np.array(meta["idx_A"], dtype=int)        # the 496-paragraph stratified test subsample used for the LLM tables
N_TRAIN_FULL = len(tr)
FULL = dict(train=min(CFG["train_n"] or len(tr), len(tr)), texts=len(dv) + len(te) + len(ch_dev) + len(ch_te))
if CFG["train_n"] and CFG["train_n"] < len(tr):   # random sample: keeps the natural class distribution (no balancing)
    tr = tr.sample(CFG["train_n"], random_state=CFG["seed"]).reset_index(drop=True)
if CFG["quick"]:
    tr = tr.head(256); dv = dv.head(200); te = te.head(200); ch_dev = ch_dev.head(200); ch_te = ch_te.head(200)
    idx_A = np.arange(100)
    CFG["out"] = CFG["out"].replace(".npz", "_QUICK.npz")
print(f"train {len(tr)} of {N_TRAIN_FULL} | dev {len(dv)} | test {len(te)} | SDGi chunks dev {len(ch_dev)} / test {len(ch_te)} | quick={CFG['quick']}")
print("train class shares, SDG 1..16:", np.round(np.bincount(tr.y.values, minlength=K) / len(tr), 3))

train 6000 of 15984 | dev 3465 | test 3566 | SDGi chunks dev 4770 / test 10464 | quick=False
train class shares, SDG 1..16: [0.053 0.034 0.081 0.104 0.111 0.056 0.075 0.039 0.063 0.036 0.06  0.024
 0.049 0.033 0.05  0.13 ]


## Step 3 - Load the model with Unsloth
The first call downloads the weights (about 5.7 GB for the 4-bit upload) and compiles kernels (a few minutes). Llama 3.1 works with `FastLanguageModel`; if that fails the cell falls back to `FastModel`.

In [ ]:
IS_GEMMA = "gemma" in CFG["model"].lower()
kw = dict(model_name=CFG["model"], max_seq_length=CFG["max_seq"], load_in_4bit=CFG["load_in_4bit"],
          load_in_16bit=not CFG["load_in_4bit"], full_finetuning=False)
try:
    model, tokenizer = FastLanguageModel.from_pretrained(**kw); LOADER = FastLanguageModel
except Exception as e:
    print("FastLanguageModel failed (", repr(e)[:200], ") -> trying FastModel")
    from unsloth import FastModel
    model, tokenizer = FastModel.from_pretrained(**kw); LOADER = FastModel
if IS_GEMMA:
    try:
        from unsloth.chat_templates import get_chat_template
        tokenizer = get_chat_template(tokenizer, chat_template="gemma-4")
    except Exception as e:
        print("could not set the gemma-4 chat template, using the built-in one:", repr(e)[:150])
tok = getattr(tokenizer, "tokenizer", tokenizer)    # vision-language models: `tokenizer` may be a processor

# precision follows what Unsloth really loaded (fp32 on a T4 for Qwen3.5 -> no autocast; fp16 -> loss scaling; bf16 -> plain autocast)
USE_BF16 = torch.cuda.is_bf16_supported()           # L4 = True (bf16), T4 = False (fp16)
AMP = torch.bfloat16 if USE_BF16 else torch.float16
USE_SCALER = not USE_BF16
def amp_ctx():
    return torch.autocast("cuda", dtype=AMP, enabled=True)
print("loaded:", getattr(model.config, "_name_or_path", "?"), "|", round(sum(p.numel() for p in model.parameters()) / 1e6), "M parameters (4-bit weights count as packed bytes)")
print("pad token:", tokenizer.pad_token if hasattr(tokenizer, "pad_token") else getattr(tok, "pad_token", None), "| BOS:", getattr(tok, "bos_token", None))
print("precision:", AMP, "| loader:", LOADER.__name__, "| GPU memory after loading: %.1f GiB" % (torch.cuda.memory_allocated() / 2**30))

==((====))==  Unsloth 2026.10.7: Fast Llama patching. Transformers: 5.17.0.
   \\   /|    NVIDIA L4. Num GPUs = 1. Max memory: 22.034 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu130. CUDA: 8.9. CUDA Toolkit: 13.0. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Unsloth: Will load unsloth/Llama-3.1-8B-Instruct-unsloth-bnb-4bit as a legacy tokenizer.


loaded: unsloth/Llama-3.1-8B-Instruct-unsloth-bnb-4bit | 4629 M parameters (4-bit weights count as packed bytes)
pad token: <|finetune_right_pad_id|> | BOS: <|begin_of_text|>
precision: torch.bfloat16 | loader: FastLanguageModel | GPU memory after loading: 5.6 GiB


## Step 4 - Prompt and label words
One answer word per SDG (index = SDG number - 1); the prompt is the same as in the Qwen and Gemma runs. We score the first token of each word *as the model would really emit it* (read from the tokenization of prompt + word), and the 16 tokens must differ.

**Llama-specific safeguard:** if two words share the same first token in Llama's tokenizer, the later word is replaced by the next alternative from `ALTS` (and its meaning is added to the prompt), and the check is repeated. For the default words the prompt text is identical to the Qwen/Gemma prompt.

In [ ]:
LABEL_WORDS = ["Poverty", "Hunger", "Health", "Education", "Gender", "Water", "Energy", "Work",
               "Industry", "Inequality", "Cities", "Consumption", "Climate", "Ocean", "Land", "Peace"]      # same default words as the Qwen/Gemma runs
ALTS = {"Poverty": ["Poor", "Destitution"], "Hunger": ["Famine", "Food"], "Health": ["Wellbeing", "Medical"], "Education": ["Schooling", "Learning"],
        "Gender": ["Women", "Sexes"], "Water": ["Sanitation", "Aqua"], "Energy": ["Power", "Electricity"], "Work": ["Jobs", "Employment"],
        "Industry": ["Infrastructure", "Innovation"], "Inequality": ["Inequalities", "Disparity", "Equity"], "Cities": ["Urban", "Settlements"],
        "Consumption": ["Production", "Waste"], "Climate": ["Warming", "Carbon"], "Ocean": ["Marine", "Sea"], "Land": ["Forests", "Biodiversity"],
        "Peace": ["Justice", "Institutions"]}
GLOSS = ["no poverty", "zero hunger", "good health and well-being", "quality education", "gender equality", "clean water and sanitation",
         "affordable and clean energy", "decent work and economic growth", "industry, innovation and infrastructure", "reduced inequalities",
         "sustainable cities and communities", "responsible consumption and production", "climate action", "life below water", "life on land",
         "peace, justice and strong institutions"]
GLOSSED = {2, 4, 5, 6, 7, 8, 9, 10, 11, 13, 14, 15}      # words whose meaning is spelled out in the prompt (as in the Qwen/Gemma prompt)

def make_prompt(words, glossed):
    gl = "; ".join(f"{words[i]} = {GLOSS[i]}" for i in sorted(glossed))
    return ("Classify the text into the single UN Sustainable Development Goal it mainly addresses. "
            "Answer with exactly one word from this list: " + ", ".join(words) + ".\n(" + gl + ".)\n\nText: {text}")

WORDS, glossed = list(LABEL_WORDS), set(GLOSSED)
PROMPT = make_prompt(WORDS, glossed)

def build_prompt(text):
    t = " ".join(str(text).split()[:CFG["max_words"]])
    msgs = [{"role": "user", "content": PROMPT.format(text=t)}]
    try:   # enable_thinking=False -> templates with a thinking mode answer directly (Llama's template ignores it)
        return tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True, enable_thinking=False)
    except TypeError:
        return tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)

def encode(texts):
  # Encode text, but hard-truncate token lists to max_seq - 5 to leave room for the prompt template
  max_tokens = CFG["max_seq"] - 10
  encoded_list = []
  for t in texts:
    ids = tok(build_prompt(t), add_special_tokens=False)["input_ids"]
    if len(ids) > max_tokens:
      ids = ids[:max_tokens]  # Hard truncation prevents sequence length mismatches
    encoded_list.append(ids)
  return encoded_list

def first_token_ids():
    global PROMPT
    PROMPT = make_prompt(WORDS, glossed)
    base = tok(build_prompt("x"), add_special_tokens=False)["input_ids"]
    ids, pieces = [], []
    for w in WORDS:
        full = tok(build_prompt("x") + w, add_special_tokens=False)["input_ids"]
        assert full[:len(base)] == base, "prompt tokens changed when the answer was appended"
        ids.append(full[len(base)]); pieces.append(len(full) - len(base))
    return ids, pieces

tried = {i: [LABEL_WORDS[i]] for i in range(K)}
for attempt in range(8):
    LABEL_IDS, n_pieces = first_token_ids()
    seen, dup = {}, []
    for i, t in enumerate(LABEL_IDS):
        if t in seen: dup.append(i)
        else: seen[t] = i
    if not dup:
        break
    for i in dup:
        nxt = [a for a in ALTS[LABEL_WORDS[i]] if a not in tried[i]]
        if not nxt:
            raise RuntimeError(f"no alternative left for '{WORDS[i]}': add more words to ALTS")
        print(f"first-token collision: '{WORDS[i]}' shares its first token with '{WORDS[seen[LABEL_IDS[i]]]}' -> renamed to '{nxt[0]}'")
        tried[i].append(nxt[0]); WORDS[i] = nxt[0]; glossed.add(i)
else:
    raise RuntimeError("could not find 16 distinct first tokens")
assert len(set(LABEL_IDS)) == K
print("answer words :", WORDS, "(default words)" if WORDS == LABEL_WORDS else "(some words were renamed: see above)")
print("label tokens :", [tok.decode([i]) for i in LABEL_IDS])
print("tokens/word  :", n_pieces, "(1 = single token; more = only the first piece is scored: fine after fine-tuning, a slight handicap for zero-shot)")
print("end of an example prompt (the answer is generated right after this):\n", repr(build_prompt("Example text.")[-90:]))
LABEL_IDS_T = torch.tensor(LABEL_IDS, device="cuda")

answer words : ['Poverty', 'Hunger', 'Health', 'Education', 'Gender', 'Water', 'Energy', 'Work', 'Industry', 'Inequality', 'Cities', 'Consumption', 'Climate', 'Ocean', 'Land', 'Peace'] (default words)
label tokens : ['Po', 'H', 'Health', 'Education', 'Gender', 'Water', 'Energy', 'Work', 'Industry', 'In', 'Cities', 'Consum', 'Climate', 'Ocean', 'Land', 'Peace']
tokens/word  : [2, 2, 1, 1, 1, 1, 1, 1, 1, 2, 1, 2, 1, 1, 1, 1] (1 = single token; more = only the first piece is scored: fine after fine-tuning, a slight handicap for zero-shot)
end of an example prompt (the answer is generated right after this):
 'stitutions.)\n\nText: Example text.<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n\n'


## Step 5 - Scoring helpers (memory-safe)
Left padding + `logits_to_keep=1` (logits only for the last position). Batches are limited by a **token budget** and processed **longest first**, so an out-of-memory problem shows up in the first minute; a batch that does not fit is split in two automatically. Some SDGi chunks are garbled and run to thousands of tokens.

In [ ]:
PAD = tok.pad_token_id if tok.pad_token_id is not None else tok.eos_token_id

def collate(seqs):
    L = max(len(s) for s in seqs)
    ids = torch.full((len(seqs), L), PAD, dtype=torch.long); att = torch.zeros((len(seqs), L), dtype=torch.long)
    for i, s in enumerate(seqs):
        ids[i, L - len(s):] = torch.tensor(s); att[i, L - len(s):] = 1
    return ids.cuda(), att.cuda()

def label_logits(ids, att):
    try:
        out = model(input_ids=ids, attention_mask=att, use_cache=False, logits_to_keep=1)
    except TypeError:
        out = model(input_ids=ids, attention_mask=att, use_cache=False, num_logits_to_keep=1)
    return out.logits[:, -1, :][:, LABEL_IDS_T]          # (batch, 16)

@torch.no_grad()
def predict(texts, desc="scoring", max_tokens=12000):
    model.eval()
    seqs = encode(list(texts)); order = np.argsort([-len(s) for s in seqs]); lens = [len(s) for s in seqs]
    print(f"{desc}: {len(seqs)} texts | longest {max(lens)} tokens | median {int(np.median(lens))}")
    out = np.zeros((len(seqs), K), dtype=np.float32)

    def run(idx):
        ok = False
        try:
            ids, att = collate([seqs[j] for j in idx])
            with amp_ctx():
                lg = label_logits(ids, att)
            out[idx] = torch.softmax(lg.float(), dim=-1).cpu().numpy(); ok = True
        except torch.cuda.OutOfMemoryError:
            pass                                          # handled outside the except block so the traceback can be freed
        if not ok:
            ids = att = lg = None; gc.collect(); torch.cuda.empty_cache()
            if len(idx) == 1:
                raise RuntimeError("a single sequence does not fit in memory")
            h = len(idx) // 2; run(idx[:h]); run(idx[h:])

    pbar = tqdm(total=len(order), desc=desc); i = 0
    while i < len(order):
        b = max(1, min(CFG["eval_bs"], max_tokens // len(seqs[order[i]])))
        idx = order[i:i + b]; run(idx); i += len(idx); pbar.update(len(idx))
    pbar.close(); return out

def report(name, P, y):
    print(f"{name}: accuracy {accuracy_score(y, P.argmax(1)):.4f} | macro-F1 {f1_score(y, P.argmax(1), average='macro', zero_division=0):.4f}")

## Step 6 - Zero-shot baseline of the SAME model (before any training)
Same prompt, same scoring, same 496 test paragraphs as the LLM tables: a clean before/after.

In [ ]:
Z = predict(te.text.iloc[idx_A].tolist(), desc="zero-shot")
y_sub = te.y.values[idx_A]
report(f"ZERO-SHOT {CFG['model']} on the test subsample (n={len(idx_A)})", Z, y_sub)
zs_acc = accuracy_score(y_sub, Z.argmax(1))
if zs_acc < 0.30:
    print(f"WARNING: zero-shot accuracy {zs_acc:.3f} is near chance (0.06-0.10). The prompt or the label tokens are probably wrong: "
          "look at the 'label tokens' and 'end of an example prompt' lines of Step 4 before training anything.")

zero-shot: 496 texts | longest 413 tokens | median 286


zero-shot:   0%|          | 0/496 [00:00<?, ?it/s]

ZERO-SHOT unsloth/Llama-3.1-8B-Instruct on the test subsample (n=496): accuracy 0.5867 | macro-F1 0.6006


## Step 6b - Padding-invariance test (about 1 minute)
Our batches are left-padded. This test scores 64 test paragraphs (a) in padded batches of 8 that alternate very short and very long texts, and (b) one by one without any padding. Llama is a standard-attention model, so agreement should be about 100%; a clearly lower value would mean the answers depend on padding.

In [ ]:
_i = np.random.default_rng(0).choice(len(te), size=min(64, len(te)), replace=False)
_seqs = encode(te.text.iloc[_i].tolist()); _lens = np.array([len(s) for s in _seqs]); _ord = np.argsort(_lens)
_mixed = [int(j) for pair in zip(_ord[:len(_ord) // 2], _ord[::-1][:len(_ord) // 2]) for j in pair]   # alternate shortest / longest

@torch.no_grad()
def _score(idx_list):
    model.eval(); ids, att = collate([_seqs[j] for j in idx_list])
    with amp_ctx():
        lg = label_logits(ids, att)
    return torch.softmax(lg.float(), dim=-1).cpu().numpy()

P_b = np.zeros((len(_seqs), K), np.float32); P_s = np.zeros_like(P_b); pad_fracs = []
for s in range(0, len(_mixed), 8):
    b = _mixed[s:s + 8]; P_b[b] = _score(b)
    pad_fracs.append(1 - sum(_lens[j] for j in b) / (len(b) * max(_lens[j] for j in b)))
for j in range(len(_seqs)):
    P_s[j] = _score([j])[0]
agree = float((P_b.argmax(1) == P_s.argmax(1)).mean())
print(f"mean padding share in the padded batches: {np.mean(pad_fracs):.0%}")
print(f"argmax agreement padded vs unpadded: {agree:.3f} | mean |prob diff|: {np.abs(P_b - P_s).mean():.4f} | max: {np.abs(P_b - P_s).max():.3f}")
print("OK: scores do not depend on padding." if agree >= 0.97 else "WARNING: scores depend on padding. Do not train: the answers depend on padding.")

mean padding share in the padded batches: 10%
argmax agreement padded vs unpadded: 0.969 | mean |prob diff|: 0.0017 | max: 0.105


## Step 7 - Add LoRA adapters (Unsloth)
`r` = adapter rank (higher = more capacity, more VRAM). `use_gradient_checkpointing="unsloth"` trades a little speed for much less memory.

In [ ]:
peft_kw = dict(r=CFG["rank"], target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
               lora_alpha=CFG["rank"], lora_dropout=0, bias="none", use_gradient_checkpointing="unsloth",
               random_state=CFG["seed"], max_seq_length=CFG["max_seq"])
if LOADER is not FastLanguageModel:      # FastModel (used in the Gemma 4 examples): train the language layers only
    peft_kw.update(finetune_vision_layers=False, finetune_language_layers=True,
                   finetune_attention_modules=True, finetune_mlp_modules=True)
model = LOADER.get_peft_model(model, **peft_kw)
params = [p for p in model.parameters() if p.requires_grad]
print("LoRA weight dtypes before:", {str(p.dtype) for p in params})
if CFG["lora_fp32"]:
    for p in params:
        p.data = p.data.float()      # LoRA weights and AdamW states in fp32
print("LoRA weight dtypes now   :", {str(p.dtype) for p in params})
n_trainable = sum(p.numel() for p in params); n_total = sum(p.numel() for p in model.parameters())
print(f"trainable parameters: {n_trainable/1e6:.1f} M of {n_total/1e6:.0f} M ({100*n_trainable/n_total:.2f}%)")

Unsloth 2026.10.7 patched 32 layers with 32 QKV layers, 32 O layers and 32 MLP layers.


LoRA weight dtypes before: {'torch.float32'}
LoRA weight dtypes now   : {'torch.float32'}
trainable parameters: 41.9 M of 4671 M (0.90%)


## Step 8 - Train (plain PyTorch loop, cross-entropy over the 16 label logits)
After every epoch the model is scored on a **random** dev sample (never on test). The best epoch (dev macro-F1) is kept and restored at the end; its LoRA tensors are also saved to `/content/best_lora_params.pt` as a safety copy.

In [ ]:
from torch.optim.lr_scheduler import LambdaLR
tr_seqs = encode(tr.text.tolist()); tr_y = torch.tensor(tr.y.values, dtype=torch.long)
n, bs, acc = len(tr_seqs), CFG["bs"], CFG["grad_acc"]
steps_per_epoch = math.ceil(n / (bs * acc)); total_steps = steps_per_epoch * CFG["epochs"]; warm = max(1, int(0.05 * total_steps))
opt = torch.optim.AdamW(params, lr=CFG["lr"], weight_decay=0.01)
sched = LambdaLR(opt, lambda s: (s + 1) / warm if s < warm else max(0.0, (total_steps - s) / max(1, total_steps - warm)))
scaler = torch.amp.GradScaler("cuda", enabled=USE_SCALER)
try:
    LOADER.for_training(model)
except Exception as e:
    print("for_training not needed:", repr(e)[:100])

d_sub = dv.sample(min(1000, len(dv)), random_state=CFG["seed"])     # RANDOM dev sample (dv.head() would be a biased slice)
best_f1, best_acc, best_ep, best_state = -1.0, 0.0, 0, None
model.train(); torch.cuda.reset_peak_memory_stats()
t_train0 = time.time(); bad = 0; step = 0; run_losses = []; log_every = 10 if CFG["quick"] else 25
for ep in range(CFG["epochs"]):
    perm = np.random.permutation(n); pbar = tqdm(range(0, n, bs * acc), desc=f"epoch {ep + 1}/{CFG['epochs']}")
    for s0 in pbar:
        chunk = perm[s0:s0 + bs * acc]; loss_tot = 0.0
        for m0 in range(0, len(chunk), bs):
            mb = chunk[m0:m0 + bs]
            ids, att = collate([tr_seqs[j] for j in mb])
            with amp_ctx():
                lg = label_logits(ids, att)
            loss = F.cross_entropy(lg.float(), tr_y[mb].cuda()) * len(mb) / len(chunk)
            scaler.scale(loss).backward(); loss_tot += loss.item()
        scaler.unscale_(opt); gn = torch.nn.utils.clip_grad_norm_(params, 1.0)
        if torch.isfinite(gn) or scaler.is_enabled():
            scaler.step(opt)
        else:
            bad += 1                                  # NaN/inf gradient in bf16/fp32: skip this step
        scaler.update(); opt.zero_grad(set_to_none=True); sched.step()
        pbar.set_postfix(loss=f"{loss_tot:.3f}")
        step += 1; run_losses.append(loss_tot)
        if step % log_every == 0:
            m = float(np.mean(run_losses[-log_every:]))
            pbar.write(f"step {step}: mean loss {m:.3f} (chance = {math.log(K):.2f}) | grad norm {float(gn):.2f} | lr {sched.get_last_lr()[0]:.2e} | skipped {bad}")
            if step == 150 and m > 2.4 and not CFG["quick"]:
                raise RuntimeError("The loss is still near chance after 150 steps: the model is not learning. Stop here and check the LoRA dtypes / label tokens.")
    Pd = predict(d_sub.text.tolist(), desc="dev")
    report(f"epoch {ep + 1}/{CFG['epochs']}: dev ({len(d_sub)} random paragraphs)", Pd, d_sub.y.values)
    f1d = f1_score(d_sub.y.values, Pd.argmax(1), average="macro", zero_division=0)
    if accuracy_score(d_sub.y.values, Pd.argmax(1)) < 0.30 and not CFG["quick"]:
        raise RuntimeError("Dev accuracy after this epoch is near chance: training failed. Do not continue to the next epoch.")
    if f1d > best_f1:
        best_f1, best_acc, best_ep = f1d, accuracy_score(d_sub.y.values, Pd.argmax(1)), ep + 1
        best_state = [p.detach().cpu().clone() for p in params]; torch.save(best_state, "/content/best_lora_params.pt")
    model.train()
train_time = time.time() - t_train0; peak_gpu = torch.cuda.max_memory_allocated() / 2**30
with torch.no_grad():                                 # keep the best epoch
    for p, t in zip(params, best_state):
        p.copy_(t.to(p.device, p.dtype))
print(f"training done in {train_time / 60:.1f} min | peak GPU memory {peak_gpu:.1f} GiB | skipped steps: {bad}")
print(f"restored the best epoch ({best_ep}): dev accuracy {best_acc:.4f} | macro-F1 {best_f1:.4f}")

epoch 1/1:   0%|          | 0/375 [00:00<?, ?it/s]

Unsloth: Will smartly offload gradients to save VRAM!
Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.
step 25: mean loss 0.921 (chance = 2.77) | grad norm 5.45 | lr 1.96e-04 | skipped 0
step 50: mean loss 0.458 (chance = 2.77) | grad norm 7.55 | lr 1.82e-04 | skipped 0
step 75: mean loss 0.448 (chance = 2.77) | grad norm 1.43 | lr 1.68e-04 | skipped 0
step 100: mean loss 0.466 (chance = 2.77) | grad norm 4.45 | lr 1.54e-04 | skipped 0
step 125: mean loss 0.483 (chance = 2.77) | grad norm 4.24 | lr 1.40e-04 | skipped 0
step 150: mean loss 0.367 (chance = 2.77) | grad norm 2.50 | lr 1.26e-04 | skipped 0
step 175: mean loss 0.365 (chance = 2.77) | grad norm 2.33 | lr 1.12e-04 | skipped 0
step 200: mean loss 0.250 (chance = 2.77) | grad norm 3.71 | lr 9.80e-05 | skipped 0
step 225: mean loss 0.349 (chance = 2.77) | grad norm 4.10 | lr 8.40e-05 | skipped 0
step 250: mean loss 0.342 (chance = 2.77) | grad norm 1.64 | lr 7.00e-05 | skipped 0
step 275: mean loss 0

dev:   0%|          | 0/1000 [00:00<?, ?it/s]

epoch 1/1: dev (1000 random paragraphs): accuracy 0.9070 | macro-F1 0.8881
training done in 42.4 min | peak GPU memory 7.7 GiB | skipped steps: 0
restored the best epoch (1): dev accuracy 0.9070 | macro-F1 0.8881


## Step 8b - Save the LoRA adapter NOW (and keep the downloaded zip)
This is the trained model. Never call `merge_and_unload()`: the scores must come from the model with the adapter attached.

In [ ]:
ADAPTER = Path(CFG["out"]).with_suffix("").name + "_adapter"
model.save_pretrained(ADAPTER); tok.save_pretrained(ADAPTER)
json.dump({"base": CFG["model"], "label_words": WORDS, "glossed": sorted(glossed), "max_words": CFG["max_words"]}, open(Path(ADAPTER) / "prompt_config.json", "w"))   # the exact answer words used
shutil.make_archive(ADAPTER, "zip", ADAPTER)
print("adapter saved:", ADAPTER + ".zip", f"{os.path.getsize(ADAPTER + '.zip') / 1e6:.0f} MB")
if not CFG["quick"]:
    files.download(ADAPTER + ".zip")

Unsloth: Restored added_tokens_decoder metadata in llama31_8b_lora_osdg_adapter/tokenizer_config.json.


adapter saved: llama31_8b_lora_osdg_adapter.zip 158 MB


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Step 9 - Score everything and save
OSDG test and dev (Setting A) and SDGi dev/test chunks (OSDG-trained model transferred to SDGi, Setting B). Rows keep the order of the exported tables, so V9 can register them like the RoBERTa scores.
The OSDG test accuracy is checked first: if it is far below the dev accuracy, something is wrong and the long SDGi scoring is not started (set `FORCE_CONTINUE = True` to override).

In [ ]:
FORCE_CONTINUE = False
torch.cuda.empty_cache(); times = {}
def timed(name, texts):
    t0 = time.time(); P = predict(texts, desc=name); times[name] = (time.time() - t0, len(texts))
    np.save(f"/content/P_{name}.npy", P); return P            # every finished set is saved to disk immediately

P_te = timed("osdg_test", te.text.tolist())
report("FINE-TUNED, OSDG test", P_te, te.y.values)
report(f"FINE-TUNED, same {len(idx_A)}-paragraph subsample", P_te[idx_A], y_sub)
acc_te = accuracy_score(te.y.values, P_te.argmax(1))
if (not CFG["quick"]) and acc_te < best_acc - 0.10 and not FORCE_CONTINUE:
    raise RuntimeError(f"OSDG test accuracy {acc_te:.3f} is far below the dev accuracy {best_acc:.3f}: something is wrong. "
                       "Do not run the long SDGi scoring; tell me the numbers. (FORCE_CONTINUE = True overrides.)")
P_dv  = timed("osdg_dev", dv.text.tolist())
P_chd = timed("sdgi_dev_chunks", ch_dev.chunk_text.tolist())
P_cht = timed("sdgi_test_chunks", ch_te.chunk_text.tolist())

infer_s_per_1k = times["osdg_test"][0] / times["osdg_test"][1] * 1000
out = Path(CFG["out"]); out.parent.mkdir(parents=True, exist_ok=True)
np.savez(out, osdg_dev=P_dv, osdg_test=P_te, sdgi_dev_ch=P_chd, sdgi_test_ch=P_cht, osdg_test_zero_sub=Z, idx_A=idx_A,
         params=n_total, trainable_params=n_trainable, train_time=train_time, infer_s_per_1k=infer_s_per_1k, peak_gpu=peak_gpu,
         model=CFG["model"], device=torch.cuda.get_device_name(0), n_train=n, epochs=CFG["epochs"], quick=CFG["quick"],
         best_epoch=best_ep, dev_macro_f1=best_f1, label=CFG["label"], label_words=np.array(WORDS), glossed=np.array(sorted(glossed)))
print("saved", out.resolve())
if CFG["quick"]:
    t_inf = sum(t for t, _ in times.values()) / sum(c for _, c in times.values())
    est = FULL["train"] * CFG["epochs"] * (train_time / max(1, n * CFG["epochs"])) + FULL["texts"] * t_inf
    print(f"QUICK RUN OK. Rough projected time of the full run with these settings: ~{est / 3600:.1f} h. "
          "Now set quick=False, restart the session and run from the imports cell down.")
else:
    files.download(CFG["out"])

osdg_test: 3566 texts | longest 450 tokens | median 292


osdg_test:   0%|          | 0/3566 [00:00<?, ?it/s]

FINE-TUNED, OSDG test: accuracy 0.8990 | macro-F1 0.8662
FINE-TUNED, same 496-paragraph subsample: accuracy 0.8448 | macro-F1 0.8397
osdg_dev: 3465 texts | longest 449 tokens | median 291


osdg_dev:   0%|          | 0/3465 [00:00<?, ?it/s]

sdgi_dev_chunks: 4770 texts | longest 3010 tokens | median 376


sdgi_dev_chunks:   0%|          | 0/4770 [00:00<?, ?it/s]

Unsloth: Input IDs of shape torch.Size([3, 3010]) with length 3010 > the model's max sequence length of 768.
We shall truncate it ourselves. It's imperative if you correct this issue first.


RuntimeError: The size of tensor a (768) must match the size of tensor b (3010) at non-singleton dimension 3

In [ ]:
# 1. Fix build_prompt so even garbled PDF chunks without spaces never exceed max_seq (768 tokens),
#    while ALWAYS keeping the closing chat template header intact at the end!
def build_prompt(text):
    # Cap both words (200) and raw characters (1400 chars ~= 350 tokens max) so prompt + template is always < 768 tokens
    t = " ".join(str(text).split()[:CFG["max_words"]])[:1400]
    msgs = [{"role": "user", "content": PROMPT.format(text=t)}]
    try:
        return tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True, enable_thinking=False)
    except TypeError:
        return tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)

# 2. Resume Step 9 from sdgi_dev_chunks (reusing P_te and P_dv already in memory / on disk!)
torch.cuda.empty_cache()
if "P_te" not in globals():
    P_te = np.load("/content/P_osdg_test.npy")
if "P_dv" not in globals():
    P_dv = np.load("/content/P_osdg_dev.npy")

report("FINE-TUNED, OSDG test", P_te, te.y.values)
report(f"FINE-TUNED, same {len(idx_A)}-paragraph subsample", P_te[idx_A], y_sub)

P_chd = timed("sdgi_dev_chunks", ch_dev.chunk_text.tolist())
P_cht = timed("sdgi_test_chunks", ch_te.chunk_text.tolist())

infer_s_per_1k = times["osdg_test"][0] / times["osdg_test"][1] * 1000
out = Path(CFG["out"]); out.parent.mkdir(parents=True, exist_ok=True)
np.savez(out, osdg_dev=P_dv, osdg_test=P_te, sdgi_dev_ch=P_chd, sdgi_test_ch=P_cht, osdg_test_zero_sub=Z, idx_A=idx_A,
         params=n_total, trainable_params=n_trainable, train_time=train_time, infer_s_per_1k=infer_s_per_1k, peak_gpu=peak_gpu,
         model=CFG["model"], device=torch.cuda.get_device_name(0), n_train=n, epochs=CFG["epochs"], quick=CFG["quick"],
         best_epoch=best_ep, dev_macro_f1=best_f1, label=CFG["label"], label_words=np.array(WORDS), glossed=np.array(sorted(glossed)))
print("saved", out.resolve())
files.download(CFG["out"])

FINE-TUNED, OSDG test: accuracy 0.8990 | macro-F1 0.8662
FINE-TUNED, same 496-paragraph subsample: accuracy 0.8448 | macro-F1 0.8397
sdgi_dev_chunks: 4770 texts | longest 758 tokens | median 376


sdgi_dev_chunks:   0%|          | 0/4770 [00:00<?, ?it/s]

sdgi_test_chunks: 10464 texts | longest 758 tokens | median 389


sdgi_test_chunks:   0%|          | 0/10464 [00:00<?, ?it/s]

saved /content/llama31_8b_lora_osdg.npz


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Step 9b - Score the robustness sets of V9_2 (Tables 1e and 1f)
V9_2 needs this model's scores on (a) the **unfiltered** OSDG test split (label-noise slices) and (b) the perturbed paragraphs. They are scored here, with the model that is still loaded, and saved to `<name>_extra.npz` (with fingerprints of the texts, which V9_2 verifies). Skipped in a quick run, or if `qwen_export.zip` does not come from V9_2 (no `robust_*.parquet`).

In [ ]:
import hashlib
rp_all, rp_pert = EXP / "robust_osdg_test_all.parquet", EXP / "robust_pert.parquet"
if CFG["quick"]:
    print("quick run: robustness scoring skipped")
elif not (rp_all.exists() and rp_pert.exists()):
    print("robust_osdg_test_all.parquet / robust_pert.parquet not found: the export must come from V9_2 (section 5b). Re-export, upload and run this cell again.")
else:
    t_all = pd.read_parquet(rp_all).text.astype(str).tolist(); t_pert = pd.read_parquet(rp_pert).text.astype(str).tolist()
    md5 = lambda xs: hashlib.md5("\n".join(xs).encode()).hexdigest()
    H_ALL, H_PERT = md5(t_all), md5(t_pert)
    for k, h in [("all_md5", H_ALL), ("pert_md5", H_PERT)]:
        if k in meta:
            assert meta[k] == h, f"{k} in meta.json does not match the parquet file: re-export from V9_2"
    t0 = time.time()
    S_pert = predict(t_pert, desc="perturbations"); np.save("/content/P_pert.npy", S_pert); files.download("/content/P_pert.npy")   # saved in Colab AND downloaded to your computer
    S_all = predict(t_all, desc="osdg_test_all"); np.save("/content/P_osdg_test_all.npy", S_all); files.download("/content/P_osdg_test_all.npy")
    assert S_all.shape == (len(t_all), K) and S_pert.shape == (len(t_pert), K)
    extra_out = CFG["out"].replace(".npz", "_extra.npz")
    np.savez(extra_out, osdg_test_all=S_all, pert=S_pert, all_md5=H_ALL, pert_md5=H_PERT, model=CFG["model"])
    print(f"saved {extra_out} | {S_all.shape} + {S_pert.shape} | {(time.time() - t0) / 60:.1f} min")
    files.download(extra_out)

perturbations: 600 texts | longest 419 tokens | median 291


perturbations:   0%|          | 0/600 [00:00<?, ?it/s]

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

osdg_test_all: 5374 texts | longest 450 tokens | median 290


osdg_test_all:   0%|          | 0/5374 [00:00<?, ?it/s]

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

saved llama31_8b_lora_osdg_extra.npz | (5374, 16) + (600, 16) | 10.1 min


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>